In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif']

In [ ]:
results_dir = Path("../data/benchmark_results")
result_files = sorted(results_dir.glob("slate_generation_benchmark_*.json"))
if not result_files:
    raise FileNotFoundError(f"No benchmark results found in {results_dir}")

result_path = result_files[-1]
print(f"Loading: {result_path.name}")
with open(result_path, 'r') as f:
    data = json.load(f)

setup = data["setup"]
print(f"I={setup['n_items']}, d={setup['d']}, K={setup['k']}, "
      f"S={setup['S']}")
print(f"Repeats: {setup['n_repeats']}")

In [ ]:
METHOD_LABELS = {
    "direct_topk": "Top-k MNL",
    "mean_topk": "Top-k MXL",
    "top1_per_sample": "Pref. Top-1",
    "greedy": "Greedy",
    "topk_greedy": "Top-k Greedy",
}

k = setup["k"]
rows = []
for name, label in METHOD_LABELS.items():
    m = data["methods"][name]
    score = np.array(m["score_phase_sec"]) * 1000
    select = np.array(m["select_phase_sec"]) * 1000
    rows.append({
        "Method": label,
        "Mean Score (ms)": np.mean(score),
        "Mean Sel/slot (ms)": np.mean(select) / k,
    })
df = pd.DataFrame(rows).set_index("Method")
df

In [ ]:
labels = df.index.tolist()
score_us = df["Mean Score (ms)"].values * 1000
sel_per_slot_us = df["Mean Sel/slot (ms)"].values * 1000

n = len(labels)
spacing = 0.7
bar_h = 0.55
y = np.arange(n) * spacing

fig, ax = plt.subplots(figsize=(4, 2.8))

ax.barh(y, score_us, height=bar_h, color='tab:blue')
ax.barh(y, sel_per_slot_us, height=bar_h, left=score_us, color='tab:orange')

for i in range(n):
    total = score_us[i] + sel_per_slot_us[i]
    fixed_label = f'{score_us[i]:.0f} µs'
    ax.text(score_us[i] / 2, y[i], fixed_label,
            ha='center', va='center', fontsize=11, fontweight='bold',
            color='white')
    if sel_per_slot_us[i] > 500:
        ax.text(score_us[i] + sel_per_slot_us[i] / 2, y[i],
                f'{sel_per_slot_us[i]:.0f} µs',
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white')
    else:
        ax.text(total + 10, y[i], f'{sel_per_slot_us[i]:.0f} µs',
                ha='left', va='center', fontsize=11)

ax.set_xticks([])
ax.set_xticklabels([])
ax.grid(False)

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=11)
ax.tick_params(axis='y', length=0)
ax.invert_yaxis()

ax.spines['bottom'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_bounds(y[0] - spacing / 2, y[-1] + spacing / 2)

score_end = score_us[-1]
total_last = score_end + sel_per_slot_us[-1]

brace_y = y[-1] + spacing * 0.55
bracket_drop = spacing * 0.22
text_y = brace_y + bracket_drop + spacing * 0.08

ax.plot([0, 0, score_end, score_end],
        [brace_y, brace_y + bracket_drop, brace_y + bracket_drop, brace_y],
        color='0.3', lw=1.0, clip_on=False)
ax.text(score_end / 2 - 60, text_y,
        'Fixed cost\n(sampling + scoring)',
        ha='center', va='top', fontsize=9, color='0.3')

ax.plot([score_end, score_end, total_last, total_last],
        [brace_y, brace_y + bracket_drop, brace_y + bracket_drop, brace_y],
        color='0.3', lw=1.0, clip_on=False)
ax.text(score_end + sel_per_slot_us[-1] / 2 + 40, text_y,
        'Cost per slot\n(item selection)',
        ha='center', va='top', fontsize=9, color='0.3')

ax.set_ylim(y[-1] + spacing * 1.1, y[0] - spacing / 2)

fig.tight_layout()
plt.savefig('../data/plots/slate_generation_timing_plot.pdf', bbox_inches='tight')
plt.show()